In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(1,  '/home/hice1/mchen439/scratch/EEG-Foundation-Model/Preprocessing')
sys.path.insert(1,  '/home/hice1/mchen439/scratch/EEG-Foundation-Model/GraphCreation')

import mne
import pandas as pd
import numpy as np
np.set_printoptions(threshold=np.inf)
np.set_printoptions(linewidth=np.inf)
from mne_bids import BIDSPath, read_raw_bids, write_raw_bids
import tsfel
cfg = tsfel.get_features_by_domain()

from readEEG import readEEG
from preprocessingPipeline import simplePipeline
from GenerateConnectivityGraphs import createDistanceMatrix

In [2]:
subjects = ["NDARAC904DMU"]

In [ ]:
HBN_patient_epochs = []
for subject in subjects:
	hbn_bids_path = BIDSPath(root="../SampleData/HBN", session=None, datatype="eeg")
	hbn_bids_path = hbn_bids_path.update(subject=subject, task="RestingState", suffix="eeg")
	HBN_patient = readEEG(hbn_bids_path, dataset="HBN")
	HBN_patient_epochs.append(simplePipeline(HBN_patient))
	print("DATA:", subject, HBN_patient.n_times, len(HBN_patient.ch_names), HBN_patient.info['sfreq'])

/home/hice1/mchen439/scratch/EEG-Foundation-Model/Preprocessing/readEEG.py:34: RuntimeWarning: Data will be preloaded. preload=False or a string preload is not supported when the data is stored in the .set file
  raw = read_raw_bids(bidsPath, extra_params={'verbose':False}, verbose=False)
/home/hice1/mchen439/scratch/EEG-Foundation-Model/Preprocessing/readEEG.py:34: RuntimeWarning: Unable to map the following column(s) to to MNE:
release_number: R1
ehq_total: 71.17
commercial_use: Yes
full_pheno: Yes
p_factor: -0.603
attention: -0.446
internalizing: 1.248
externalizing: 0.325
RestingState: available
DespicableMe: available
FunwithFractals: available
ThePresent: available
DiaryOfAWimpyKid: available
contrastChangeDetection_1: available
contrastChangeDetection_2: available
contrastChangeDetection_3: available
surroundSupp_1: available
surroundSupp_2: available
seqLearning6target: unavailable
seqLearning8target: available
symbolSearch: available
  raw = read_raw_bids(bidsPath, extra_param

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 50 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 50.00 Hz
- Upper transition bandwidth: 12.50 Hz (-6 dB cutoff frequency: 56.25 Hz)
- Filter length: 3301 samples (6.602 s)



/home/hice1/mchen439/scratch/EEG-Foundation-Model/Preprocessing/readEEG.py:48: RuntimeWarning: Fiducial point nasion not found, assuming identity unknown to head transformation
  raw.set_montage(montage)
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.3s


Not setting metadata
403 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 403 events and 128 original time points ...
0 bad epochs dropped
Not setting metadata
80 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 80 events and 640 original time points ...
0 bad epochs dropped
Not setting metadata
26 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 26 events and 1920 original time points ...
0 bad epochs dropped
Not setting metadata
6 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 6 events and 7680 original time points ...
0 bad epochs dropped
Dropped 140 epochs: 0, 1, 2, 3, 4, 5, 6, 7, 11, 14, 23, 27, 34, 42, 43, 44, 45, 64, 66, 79, 81, 93, 94, 95, 111, 112, 113, 114, 115, 116, 117, 140, 141, 142, 144, 145, 146, 152, 157,

In [12]:
for i in range(len(HBN_patient_epochs)):
	minute_epochs = HBN_patient_epochs[i][3]
	for epoch in minute_epochs:
		features = tsfel.time_series_features_extractor(cfg, epoch, fs=128)

KeyboardInterrupt: 

In [31]:
for i in range(len(HBN_patient_epochs)):
	info = HBN_patient_epochs[i][0].info
	dist_matrix = createDistanceMatrix(info)
	print(dist_matrix.shape, dist_matrix.min(), dist_matrix.max())

(21, 21) 0.0 1.0
(21, 21) 0.0 1.0
(21, 21) 0.0 1.0


In [1]:
import mne
import gc
import numpy as np
import pandas as pd
import os, argparse
from pathlib import Path
from mne_bids import BIDSPath, read_raw_bids
from tqdm import tqdm
from preprocessingPipeline import simplePipeline
from concurrent.futures import ProcessPoolExecutor
import warnings
import traceback

HBN_ELECTRODE_MAP = {
	'Fp1': 'E22',
	'Fp2': 'E9',
	'F7': 'E33',
	'F3': 'E24',
	'Fz': 'E11',
	'F4': 'E124',
	'F8': 'E122',
	'T3': 'E45',
	'C3': 'E36',
	'Cz': 'Cz',
	'C4': 'E104',
	'T4': 'E108',
	'T5': 'E58',
	'P3': 'E52',
	'Pz': 'E62',
	'P4': 'E92',
	'T6': 'E96',
	'O1': 'E70',
	'O2': 'E83',
}

ELECTRODES = [f'E{i}' for i in range(1, 129)]
TO_DROP = [electrode for electrode in ELECTRODES if electrode not in HBN_ELECTRODE_MAP.values()]
HBN_ELECTRODE_MAP_REVERSED = dict((v,k) for k,v in HBN_ELECTRODE_MAP.items())


ModuleNotFoundError: No module named 'readEEG'

In [ ]:
bids_path = BIDSPath(root="~/mycontainer/ds004186-download", subject="NDARAA075AM", datatype="eeg", suffix="eeg", extension=".set")
tasks = ["EC", "EO",]
runs = np.arange(1, 6)
for task in tasks:
	bids_path.update(task=task)
	file_path = os.path.join("~/TEST2", subject, "epochs", f"{task}.fif")
	task_all_data = []
	if not os.path.exists(file_path):
		runs = sorted(set([bp.run for bp in bids_path.match()]))
		for run in runs:
			print(f"Processing subject: {subject} task {task} run {run}")
			try:
				bids_path.update(run=run)		
				raw = read_raw_bids(bids_path, extra_params={'preload':True}, verbose=False)
				raw.drop_channels(TO_DROP)
				mapping = {ch: 'eeg' for ch in raw.ch_names}
				raw.set_channel_types(mapping)
				raw.rename_channels(HBN_ELECTRODE_MAP_REVERSED)
				if indices is None:
					indices = raw.ch_names
				task_all_data.append(raw)
			except Exception as e:
				print(f"Failed to process subject: {subject} for task {task} on run {run}")
		concatenated_raw = mne.concatenate_raws(task_all_data)
